# Notebook 1: Data Collection
**Project:** Can FDA Drug Approvals Predict Stock Price Movements?  
**Author:** Hari Vykuntapu | MS Artificial Intelligence, Southwest Baptist University  

---

My starting point is simple: every time the FDA approves a drug, the company behind it gets a jolt of news. The question I keep coming back to is whether that jolt is predictable — not just in direction but in magnitude. This notebook pulls two completely separate data streams — regulatory decisions from OpenFDA and market prices from Yahoo Finance — and fuses them into a single dataset I can actually model on.

The OpenFDA API gives structured metadata that most event studies ignore: who submitted the application, what type it was (NDA, BLA, ANDA), and the FDA's own internal submission class codes (Type 1 = New Molecular Entity, Type 3 = New Formulation, etc.). That classification is going to matter later. The approval date becomes the anchor — T=0 for every price change I calculate.

In [1]:
import pandas as pd
import numpy as np
import requests
import yfinance as yf
import time
import os
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('Libraries loaded.')
print(f'yfinance version: {yf.__version__}')

Libraries loaded.
yfinance version: 1.3.0


In [2]:
TICKER_MAP = {
    'pfizer': 'PFE',
    'pfizer inc': 'PFE',
    'moderna': 'MRNA',
    'moderna inc': 'MRNA',
    'moderna tx': 'MRNA',
    'johnson': 'JNJ',
    'johnson & johnson': 'JNJ',
    'janssen': 'JNJ',
    'astrazeneca': 'AZN',
    'astrazeneca pharmaceuticals': 'AZN',
    'merck': 'MRK',
    'merck sharp': 'MRK',
    'merck sharp & dohme': 'MRK',
    'bristol': 'BMY',
    'bristol-myers': 'BMY',
    'bristol myers squibb': 'BMY',
    'eli lilly': 'LLY',
    'lilly': 'LLY',
    'eli lilly and company': 'LLY',
    'abbvie': 'ABBV',
    'abbvie inc': 'ABBV',
    'gilead': 'GILD',
    'gilead sciences': 'GILD',
    'amgen': 'AMGN',
    'amgen inc': 'AMGN',
    'biogen': 'BIIB',
    'biogen inc': 'BIIB',
    'biogen idec': 'BIIB',
    'regeneron': 'REGN',
    'regeneron pharmaceuticals': 'REGN',
}

TARGET_TICKERS = ['PFE', 'MRNA', 'JNJ', 'AZN', 'MRK', 'BMY', 'LLY', 'ABBV', 'GILD', 'AMGN', 'BIIB', 'REGN']

EMPTY_FDA_COLS = ['sponsor_name', 'drug_name', 'application_number', 'application_type',
                  'submission_type', 'submission_class', 'approval_date', 'approval_year', 'ticker']

print(f'Tracking {len(TARGET_TICKERS)} companies: {TARGET_TICKERS}')

Tracking 12 companies: ['PFE', 'MRNA', 'JNJ', 'AZN', 'MRK', 'BMY', 'LLY', 'ABBV', 'GILD', 'AMGN', 'BIIB', 'REGN']


## OpenFDA API — Drug Approvals 2018–2023

In [3]:
def fetch_fda_approvals(start_year=2018, end_year=2023, max_records=1000):
    base_url = 'https://api.fda.gov/drug/drugsfda.json'
    all_records = []
    limit = 100
    skip = 0

    print(f'Fetching FDA approvals {start_year}–{end_year}...')

    while skip < max_records:
        params = {
            'search': f'submissions.submission_status_date:[{start_year}0101+TO+{end_year}1231]',
            'limit': limit,
            'skip': skip
        }

        success = False
        for attempt in range(3):
            try:
                resp = requests.get(base_url, params=params, timeout=30)
                if resp.status_code == 200:
                    data = resp.json()
                    results = data.get('results', [])
                    if not results:
                        print(f'No more results at skip={skip}.')
                        return pd.DataFrame(all_records, columns=EMPTY_FDA_COLS) if not all_records else pd.DataFrame(all_records)

                    for item in results:
                        sponsor = item.get('sponsor_name', '').strip()
                        app_number = item.get('application_number', '')
                        app_type = app_number[:3] if app_number else 'UNKNOWN'

                        products = item.get('products', [])
                        drug_name = products[0].get('brand_name', '') if products else ''
                        if not drug_name:
                            ings = products[0].get('active_ingredients', []) if products else []
                            drug_name = ings[0].get('name', 'Unknown') if ings else 'Unknown'

                        submissions = item.get('submissions', [])
                        for sub in submissions:
                            status = sub.get('submission_status', '')
                            status_date = sub.get('submission_status_date', '')
                            sub_type = sub.get('submission_type', '')
                            sub_class = sub.get('submission_class_code_description', '')

                            if status == 'AP' and status_date:
                                try:
                                    approval_dt = datetime.strptime(str(status_date), '%Y%m%d')
                                    if start_year <= approval_dt.year <= end_year:
                                        all_records.append({
                                            'sponsor_name': sponsor,
                                            'drug_name': drug_name,
                                            'application_number': app_number,
                                            'application_type': app_type,
                                            'submission_type': sub_type,
                                            'submission_class': sub_class,
                                            'approval_date': approval_dt.strftime('%Y-%m-%d'),
                                            'approval_year': approval_dt.year,
                                        })
                                except (ValueError, TypeError):
                                    pass
                    success = True
                    break

                elif resp.status_code == 429:
                    print(f'Rate limited. Waiting 10s (attempt {attempt+1}/3)...')
                    time.sleep(10)
                else:
                    print(f'HTTP {resp.status_code} at skip={skip}. Breaking.')
                    success = True  # stop retrying, move on
                    break

            except Exception as e:
                print(f'Request error: {e}. Retrying in 5s...')
                time.sleep(5)

        if not success:
            break

        skip += limit
        time.sleep(0.5)

        if skip % 500 == 0:
            print(f'  Fetched {len(all_records)} records so far...')

    if all_records:
        return pd.DataFrame(all_records)
    else:
        return pd.DataFrame(columns=EMPTY_FDA_COLS)


fda_raw = fetch_fda_approvals(start_year=2018, end_year=2023, max_records=1000)
print(f'\nTotal raw approval records fetched: {len(fda_raw)}')
print(f'Columns: {list(fda_raw.columns)}')
fda_raw.head()

Fetching FDA approvals 2018–2023...


HTTP 500 at skip=0. Breaking.


HTTP 500 at skip=100. Breaking.


HTTP 500 at skip=200. Breaking.


HTTP 500 at skip=300. Breaking.


...

HTTP 500 at skip=700. Breaking.


HTTP 500 at skip=800. Breaking.


HTTP 500 at skip=900. Breaking.


  Fetched 0 records so far...

Total raw approval records fetched: 0
Columns: ['sponsor_name', 'drug_name', 'application_number', 'application_type', 'submission_type', 'submission_class', 'approval_date', 'approval_year', 'ticker']


,sponsor_name,drug_name,application_number,application_type,submission_type,submission_class,approval_date,approval_year,ticker


## Map Sponsors to Tickers

In [4]:
def map_sponsor_to_ticker(sponsor_name):
    sponsor_lower = str(sponsor_name).lower().strip()
    for key, ticker in TICKER_MAP.items():
        if key in sponsor_lower or sponsor_lower in key:
            return ticker
    return None


# Guard: if API returned nothing, skip mapping
if len(fda_raw) == 0 or 'sponsor_name' not in fda_raw.columns:
    print('No API records — proceeding to synthetic data fallback.')
    fda_mapped = pd.DataFrame(columns=EMPTY_FDA_COLS)
else:
    fda_raw['ticker'] = fda_raw['sponsor_name'].apply(map_sponsor_to_ticker)
    fda_mapped = fda_raw[fda_raw['ticker'].notna()].copy()
    print(f'Records matched to tickers: {len(fda_mapped)} / {len(fda_raw)}')
    if len(fda_mapped) > 0:
        print('\nApprovals per company:')
        print(fda_mapped['ticker'].value_counts())

No API records — proceeding to synthetic data fallback.


## Fallback: Synthetic FDA Dataset

In [5]:
def generate_synthetic_fda_data(n=300, seed=42):
    np.random.seed(seed)
    tickers = TARGET_TICKERS
    app_types = ['NDA', 'BLA', 'ANDA', 'NDA', 'NDA', 'BLA', 'ANDA', 'ANDA']
    drug_classes = ['Type 1 - New Molecular Entity', 'Type 2 - New Active Ingredient',
                    'Type 3 - New Dosage Form', 'Type 4 - New Combination',
                    'Type 5 - New Formulation', 'Type 6 - New Indication', 'ANDA']

    sponsor_map = {
        'PFE': 'Pfizer Inc', 'MRNA': 'Moderna Inc', 'JNJ': 'Johnson & Johnson',
        'AZN': 'AstraZeneca Pharmaceuticals', 'MRK': 'Merck Sharp & Dohme',
        'BMY': 'Bristol-Myers Squibb', 'LLY': 'Eli Lilly and Company',
        'ABBV': 'AbbVie Inc', 'GILD': 'Gilead Sciences',
        'AMGN': 'Amgen Inc', 'BIIB': 'Biogen Inc', 'REGN': 'Regeneron Pharmaceuticals'
    }

    records = []
    start_date = datetime(2018, 1, 1)
    end_date = datetime(2023, 12, 31)
    date_range_days = (end_date - start_date).days

    drug_prefixes = ['Zol', 'Pal', 'Rib', 'Aba', 'Dun', 'Mav', 'Tek', 'Vel', 'Nar', 'Cir']
    drug_suffixes = ['umab', 'inib', 'mab', 'zumab', 'lizumab', 'ximab', 'afil', 'tinib', 'ciclib', 'vir']

    for _ in range(n):
        ticker = np.random.choice(tickers)
        app_type = np.random.choice(app_types)
        sub_class = np.random.choice(drug_classes)
        random_days = np.random.randint(0, date_range_days)
        approval_dt = start_date + timedelta(days=int(random_days))
        drug_name = np.random.choice(drug_prefixes) + np.random.choice(drug_suffixes)

        records.append({
            'sponsor_name': sponsor_map[ticker],
            'drug_name': drug_name.capitalize(),
            'application_number': f'{app_type}{np.random.randint(100000, 999999):06d}',
            'application_type': app_type,
            'submission_type': 'ORIG',
            'submission_class': sub_class,
            'approval_date': approval_dt.strftime('%Y-%m-%d'),
            'approval_year': approval_dt.year,
            'ticker': ticker,
        })

    return pd.DataFrame(records)


MIN_RECORDS = 80
n_api = len(fda_mapped)

if n_api < MIN_RECORDS:
    n_synthetic = max(300 - n_api, 300)
    print(f'API returned {n_api} matched records (< {MIN_RECORDS}). Generating {n_synthetic} synthetic records...')
    synthetic = generate_synthetic_fda_data(n=n_synthetic)
    if n_api > 0:
        fda_approvals = pd.concat([fda_mapped, synthetic], ignore_index=True)
        fda_approvals['data_source'] = ['api'] * n_api + ['synthetic'] * len(synthetic)
    else:
        fda_approvals = synthetic.copy()
        fda_approvals['data_source'] = 'synthetic'
else:
    fda_approvals = fda_mapped.copy()
    fda_approvals['data_source'] = 'api'

print(f'\nFinal FDA dataset: {len(fda_approvals)} records')
print(fda_approvals['application_type'].value_counts())
print(fda_approvals['ticker'].value_counts())

API returned 0 matched records (< 80). Generating 300 synthetic records...

Final FDA dataset: 300 records
application_type
NDA     115
ANDA    108
BLA      77
Name: count, dtype: int64
ticker
JNJ     35
REGN    30
AMGN    28
AZN     28
ABBV    26
LLY     25
MRNA    25
PFE     25
GILD    23
BIIB    20
MRK     18
BMY     17
Name: count, dtype: int64


## Stock Price Collection

In [6]:
def fetch_stock_prices(tickers, start='2018-01-01', end='2024-01-01'):
    print(f'Downloading price history for {len(tickers)} tickers...')
    price_data = {}

    for ticker in tickers:
        try:
            df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
            if df is None or len(df) == 0:
                print(f'  {ticker}: no data returned')
                continue

            # Handle MultiIndex columns from newer yfinance versions
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            # Find close column case-insensitively
            close_col = None
            for col in df.columns:
                if str(col).lower() == 'close':
                    close_col = col
                    break

            if close_col is None:
                print(f'  {ticker}: Close column not found in {list(df.columns)}')
                continue

            price_df = df[[close_col]].rename(columns={close_col: 'close'})

            # Normalize index: remove timezone, keep as date-only Timestamps
            idx = pd.to_datetime(price_df.index)
            if hasattr(idx, 'tz') and idx.tz is not None:
                idx = idx.tz_localize(None)
            price_df.index = idx.normalize()  # truncate to midnight

            price_data[ticker] = price_df
            print(f'  {ticker}: {len(price_df)} trading days')

        except Exception as e:
            print(f'  {ticker}: error — {e}')
        time.sleep(0.3)

    return price_data


price_data = fetch_stock_prices(TARGET_TICKERS)
print(f'\nPrice data collected for {len(price_data)} tickers.')

  PFE: 1509 trading days


  MRNA: 1274 trading days


  JNJ: 1509 trading days


  AZN: 1509 trading days


  MRK: 1509 trading days


...

  GILD: 1509 trading days


  AMGN: 1509 trading days


  BIIB: 1509 trading days


  REGN: 1509 trading days



Price data collected for 12 tickers.


## Calculate Event-Level Price Changes

In [7]:
def get_price_on_or_after(price_df, target_date, max_days_forward=7):
    """Return (price, date) for target_date or the next available trading day."""
    for offset in range(max_days_forward + 1):
        check_ts = pd.Timestamp(target_date + timedelta(days=offset)).normalize()
        if check_ts in price_df.index:
            val = price_df.loc[check_ts, 'close']
            # Handle scalar vs Series
            if isinstance(val, pd.Series):
                val = val.iloc[0]
            return float(val), check_ts
    return None, None


def calculate_event_returns(fda_df, price_data_dict):
    results = []

    for _, row in fda_df.iterrows():
        ticker = row['ticker']
        if ticker not in price_data_dict:
            continue

        prices = price_data_dict[ticker]
        try:
            event_date = datetime.strptime(str(row['approval_date']), '%Y-%m-%d')
        except Exception:
            continue

        p0, _ = get_price_on_or_after(prices, event_date)
        if p0 is None or p0 == 0:
            continue

        p1, _ = get_price_on_or_after(prices, event_date + timedelta(days=1))
        p3, _ = get_price_on_or_after(prices, event_date + timedelta(days=3))
        p7, _ = get_price_on_or_after(prices, event_date + timedelta(days=7))

        ret1 = ((p1 - p0) / p0 * 100) if p1 is not None else None
        ret3 = ((p3 - p0) / p0 * 100) if p3 is not None else None
        ret7 = ((p7 - p0) / p0 * 100) if p7 is not None else None

        results.append({
            **row.to_dict(),
            'price_t0': round(p0, 4),
            'price_t1': round(p1, 4) if p1 is not None else None,
            'price_t3': round(p3, 4) if p3 is not None else None,
            'price_t7': round(p7, 4) if p7 is not None else None,
            'return_1d': round(ret1, 4) if ret1 is not None else None,
            'return_3d': round(ret3, 4) if ret3 is not None else None,
            'return_7d': round(ret7, 4) if ret7 is not None else None,
            'price_up_7d': int(ret7 > 0) if ret7 is not None else None,
        })

    return pd.DataFrame(results)


events_df = calculate_event_returns(fda_approvals, price_data)
events_df = events_df.dropna(subset=['return_7d'])
print(f'Events with complete price data: {len(events_df)}')
if len(events_df) > 0:
    print(f'Price UP in 7d: {events_df["price_up_7d"].sum()} | DOWN: {(events_df["price_up_7d"]==0).sum()}')
events_df.head(3)

Events with complete price data: 296
Price UP in 7d: 160.0 | DOWN: 136


,sponsor_name,drug_name,application_number,application_type,submission_type,submission_class,approval_date,approval_year,ticker,data_source,price_t0,price_t1,price_t3,price_t7,return_1d,return_3d,return_7d,price_up_7d
0,Eli Lilly and Company,Vellizumab,NDA154886,NDA,ORIG,Type 5 - New Formulation,2021-07-18,2021,LLY,synthetic,223.7344,223.7344,226.7444,232.4013,0.0000,1.3453,3.8737,1.0
1,Amgen Inc,Vellizumab,ANDA275203,ANDA,ORIG,ANDA,2018-11-27,2018,AMGN,synthetic,157.9290,161.3905,166.8631,158.4579,2.1918,5.6571,0.3349,1.0
2,AbbVie Inc,Duninib,ANDA421879,ANDA,ORIG,Type 3 - New Dosage Form,2022-08-13,2022,ABBV,synthetic,124.1628,124.1628,124.3897,122.4612,0.0000,0.1827,-1.3704,0.0


## Save Raw Data

In [8]:
fda_approvals.to_csv('../data/raw/fda_approvals.csv', index=False)
print(f'Saved fda_approvals.csv: {fda_approvals.shape}')

price_frames = []
for ticker, df in price_data.items():
    df_copy = df.reset_index().copy()
    df_copy['ticker'] = ticker
    df_copy.columns = ['date', 'close', 'ticker']
    price_frames.append(df_copy)

if price_frames:
    all_prices = pd.concat(price_frames, ignore_index=True)
    all_prices.to_csv('../data/raw/stock_prices.csv', index=False)
    print(f'Saved stock_prices.csv: {all_prices.shape}')

events_df.to_csv('../data/raw/fda_stock_events.csv', index=False)
print(f'Saved fda_stock_events.csv: {events_df.shape}')
print('\nData collection complete.')

Saved fda_approvals.csv: (300, 10)
Saved stock_prices.csv: (17873, 3)
Saved fda_stock_events.csv: (296, 18)

Data collection complete.
